In [1]:
import sys
sys.path.append('..')  # Sube un nivel
from utils import *
from scipy.stats import mannwhitneyu

In [2]:
df_imputed = pd.read_csv(CFG.path_df_imputed_corrected)
print(df_imputed.columns)
print(f"Dataset shape: {df_imputed.shape}")
df_imputed.columns = clean_feature_names(df_imputed.columns)
print(f"\nColumns: {list(df_imputed.columns)}")
df_imputed.tail()

Index(['Plant_Height (cm)', 'Chlorophyll (SPAD)', 'Number of Flowers',
       'Number of Harvested Fruits', 'Weight of Harvested Fruits (Kg)',
       'Fruit Height (mm)', 'Sap pH', 'Sap K (ppm)', 'Sap Ca (ppm)',
       'Sap Na (ppm)', 'Sap NO3 (ppm)', 'Sap Conductivity (mS/cm)',
       'Soil pH Horiba', 'Soil K Horiba (ppm)', 'Soil Ca Horiba (ppm)',
       'Soil Na Horiba (ppm)', 'Soil NO3 Horiba (ppm)',
       'Soil Conductivity Horiba (mS/cm)', 'Fruit Diameter (mm)',
       '7in1_Ph[pH]', '7in1_Moisture[%RH]', '7in1_S_Temperature[C]',
       'Air_sensor_Temperature[C]', 'Air_sensor_Humidity[%RH]',
       '7in1_Conductivity[mS/cm]', 'Pynamometer_Radiation[W/m2]', 'Nitrogen',
       'Phosphorus', 'Potassium', 'Year', 'Month', 'Day', 'Treatment_Num'],
      dtype='object')
Dataset shape: (2292, 33)

Columns: ['Plant_Height (cm)', 'Chlorophyll (SPAD)', 'Number of Flowers', 'Number of Harvested Fruits', 'Weight of Harvested Fruits (Kg)', 'Fruit Height (mm)', 'Sap pH', 'Sap K (ppm)', 'Sap 

,Plant_Height (cm),Chlorophyll (SPAD),Number of Flowers,Number of Harvested Fruits,Weight of Harvested Fruits (Kg),Fruit Height (mm),Sap pH,Sap K (ppm),Sap Ca (ppm),Sap Na (ppm),...,Air_sensor_Humidity(%RH),7in1_Conductivity(mS/cm),Pynamometer_Radiation(W/m2),Nitrogen,Phosphorus,Potassium,Year,Month,Day,Treatment_Num
2287,171.0,57.33,23.6,1.0,0.180,70.3,5.438,3580.0,385.4,49.60,...,79.64,0.1307,271.1,2.0,2.0,2.0,2024.0,12.0,4.0,27.0
2288,167.2,57.19,23.4,7.0,0.706,60.7,5.438,3230.0,263.2,49.70,...,68.08,0.1242,262.7,2.0,2.0,2.0,2024.0,12.0,4.0,27.0
2289,136.6,60.03,14.1,6.0,0.829,70.2,5.506,3490.0,226.0,57.10,...,63.30,0.0280,269.0,0.0,0.0,1.0,2024.0,12.0,4.0,4.0
2290,64.6,50.57,7.7,2.0,0.174,50.7,5.667,3330.0,223.6,50.35,...,53.07,0.2098,347.2,0.0,0.0,1.0,2024.0,12.0,4.0,4.0
2291,180.5,59.77,19.7,1.0,0.150,70.1,5.594,3480.0,181.2,57.40,...,64.70,0.0144,263.5,0.0,0.0,1.0,2024.0,12.0,4.0,4.0


In [3]:
# with open(CFG.treat_quantiles_path, 'r') as f:
#     treatments_quantile_unified = json.load(f)

In [4]:
df_imputed, df_imputed['target'] = codificar_clase_cuartiles(df_imputed, filter_data=True, CFG=CFG)

In [5]:
CFG.Root = "../"
json_best_variables = Rf"{CFG.Root}\Resultados\classification_exclude_prod\most_frequent_variables_80.json"
# TODO: Checkear que la ruta existe. Se debió ejecutar el modelo de train anterior.
list_best_vars = read_best_variables(json_best_variables)

In [6]:
list_best_vars

['Soil pH Horiba',
 'Sap Na (ppm)',
 'Chlorophyll (SPAD)',
 'Sap Ca (ppm)',
 'Sap NO3 (ppm)',
 '7in1_S_Temperature(C)',
 'Sap Conductivity (mS/cm)',
 'Soil Ca Horiba (ppm)',
 'Air_sensor_Humidity(%RH)',
 'Soil Na Horiba (ppm)',
 '7in1_Ph(pH)',
 'Sap pH']

In [7]:
for var in list_best_vars:
    group1 = df_imputed[df_imputed['target'] == 0][var]
    group2 = df_imputed[df_imputed['target'] == 1][var]
    
    stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
    print(f"Variable: {var}, Mann-Whitney U statistic: {stat}, p-value: {p:.6f}")

Variable: Soil pH Horiba, Mann-Whitney U statistic: 139392.5, p-value: 0.000000
Variable: Sap Na (ppm), Mann-Whitney U statistic: 213350.0, p-value: 0.000031
Variable: Chlorophyll (SPAD), Mann-Whitney U statistic: 144999.5, p-value: 0.000000
Variable: Sap Ca (ppm), Mann-Whitney U statistic: 194543.0, p-value: 0.260613
Variable: Sap NO3 (ppm), Mann-Whitney U statistic: 169161.0, p-value: 0.002932
Variable: 7in1_S_Temperature(C), Mann-Whitney U statistic: 236400.0, p-value: 0.000000
Variable: Sap Conductivity (mS/cm), Mann-Whitney U statistic: 209563.5, p-value: 0.000384
Variable: Soil Ca Horiba (ppm), Mann-Whitney U statistic: 225147.0, p-value: 0.000000
Variable: Air_sensor_Humidity(%RH), Mann-Whitney U statistic: 151908.0, p-value: 0.000000
Variable: Soil Na Horiba (ppm), Mann-Whitney U statistic: 211078.5, p-value: 0.000147
Variable: 7in1_Ph(pH), Mann-Whitney U statistic: 171492.5, p-value: 0.009371
Variable: Sap pH, Mann-Whitney U statistic: 199108.5, p-value: 0.062534


In [8]:
df_imputed['Fecha'] = pd.to_datetime(df_imputed[['Year', 'Month', 'Day']])
results = []
alerta_generada = False
date_cosecha = pd.to_datetime("2024-10-02")

dict_alertas ={}
for var in list_best_vars:
    n_semana = 0
    n_semana_cosecha = 0
    n_semana_alerta = 0
    alerta_generada = False
    for date in df_imputed['Fecha'].unique():
        group1 = df_imputed[(df_imputed['target'] == 0) & (df_imputed['Fecha'] == date)][var]
        group2 = df_imputed[(df_imputed['target'] == 1) & (df_imputed['Fecha'] == date)][var]
        
        if len(group1) > 0 and len(group2) > 0:
            stat, p = mannwhitneyu(group1, group2, alternative='two-sided')
            results.append({'Fecha': date, 'Variable': var, 'U_statistic': stat, 'p_value': p, 'n_group1': len(group1), 'n_group2': len(group2)})
            print(f"Date: {date}, Variable: {var}, Mann-Whitney U statistic: {stat}, p-value: {p:.6f}")
        else:
            print(f"Date: {date}, Variable: {var}, Not enough data for one of the groups.")
        n_semana += 1
        if date == date_cosecha:
            n_semana_cosecha = n_semana
        if p < 0.05 and not alerta_generada:
            print(f"ALERTA: {var} ")
            alerta_generada = True
            n_semana_alerta = n_semana
    print(f"Alerta de {var} - Semana {n_semana_alerta - n_semana_cosecha}")
    dict_alertas[var] = n_semana_alerta - n_semana_cosecha
    
df_results = pd.DataFrame(results)
df_results.to_csv('mann_whitney_test.csv', index=False)
df_results.to_excel('mann_w-u-test2.xlsx', index=False)

print(dict_alertas)
#converit dict_alertas a dataframe
df_alertas = pd.DataFrame(list(dict_alertas.items()), columns=['Variable', 'Semanas_antes_cosecha'])
df_alertas.to_csv('alertas_variables.csv', index=False)

print(f"\nResultados guardados en mann_whitney_test.csv. Total de pruebas: {len(df_results)}")
df_results.head()

Date: 2024-07-30 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 42.0, p-value: 0.292162
Date: 2024-08-06 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 91.5, p-value: 0.174132
Date: 2024-08-14 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 142.0, p-value: 0.610532
Date: 2024-08-21 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 130.5, p-value: 0.939815
Date: 2024-08-27 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 40.0, p-value: 0.441803
Date: 2024-08-28 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 139.5, p-value: 0.678395
Date: 2024-09-04 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 95.0, p-value: 0.219897
Date: 2024-09-11 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 119.0, p-value: 0.268670
Date: 2024-09-17 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U statistic: 40.5, p-value: 0.698014
Date: 2024-09-25 00:00:00, Variable: Soil pH Horiba, Mann-Whitney U s

,Fecha,Variable,U_statistic,p_value,n_group1,n_group2
0,2024-07-30,Soil pH Horiba,42.0,0.292162,8,8
1,2024-08-06,Soil pH Horiba,91.5,0.174132,16,16
2,2024-08-14,Soil pH Horiba,142.0,0.610532,16,16
3,2024-08-21,Soil pH Horiba,130.5,0.939815,16,16
4,2024-08-27,Soil pH Horiba,40.0,0.441803,8,8
